# Pipeline B Notebook (Qwen3.5 + Mistral-7B Classify)

本 Notebook 固定使用两阶段流程：
1. Qwen3.5 接收 meme 图像 + OCR 文本做联合语义描述
2. Mistral-7B 基于该描述做 Task4/Task6 few-shot 判别

说明：当前数据仍是 EXIST 2024 Task4/Task6，因此结果用于方法验证而非与 EXIST2025 子任务直接横比。

In [ ]:
import importlib
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor, AutoTokenizer

from dataset import create_data_splits
import pipeline_b as pipeline_b_mod
importlib.reload(pipeline_b_mod)
from pipeline_b import (
    TASK6_LABELS,
    evaluate_pipeline_b,
    load_or_generate_descriptions,
    load_rows,
    resolve_local_model_path,
    select_prototypes_task4,
    select_prototypes_task4_full_consensus,
    select_prototypes_task6,
    select_prototypes_task6_full_consensus,
    task4_agreement,
    task6_label_agreement,
)

In [2]:
@dataclass
class Config:
    data_dir: str = "/root/QWen/EXIST 2024 Memes Dataset"
    output_dir: str = "/root/QWen/outputs/pipeline_b_qwen35_mistral"
    desc_cache_path: str = "/root/QWen/outputs/pipeline_b_qwen35_mistral/descriptions.jsonl"

    seed: int = 42
    force_split: bool = False

    # 第一阶段固定为 Qwen3.5 图文联合理解模型（不使用 Qwen2.5）
    stage1_model_name: str = "Qwen/Qwen3.5-9B"
    classifier_model_name: str = "/root/autodl-tmp/mistral_models/7B-Instruct-v0.3"
    model_cache_dir: str = "/root/autodl-tmp/hf-cache"

    # Few-shot 用小而精策略：优先 full-consensus，数量控制在 3-5 这个量级
    prototype_mode: str = "full-consensus"
    k_task4: int = 3
    k_task6: int = 1

    max_train_rows: int = 0
    max_test_rows: int = 0

    max_desc_tokens: int = 96
    eval_mode: str = "both"
    # 建议先关闭门控做同测试集对照，避免 Task4 错误级联到 Task6
    disable_hierarchical_gate: bool = True

cfg = Config()
if "2.5-vl" in cfg.stage1_model_name.lower():
    raise ValueError("当前配置禁止使用 Qwen2.5-VL，请保持 stage1_model_name 为 Qwen3.5")

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print(cfg)

Config(data_dir='/root/QWen/EXIST 2024 Memes Dataset', output_dir='/root/QWen/outputs/pipeline_b_qwen35_mistral', desc_cache_path='/root/QWen/outputs/pipeline_b_qwen35_mistral/descriptions.jsonl', seed=42, force_split=False, stage1_model_name='Qwen/Qwen3.5-9B', classifier_model_name='/root/autodl-tmp/mistral_models/7B-Instruct-v0.3', model_cache_dir='/root/autodl-tmp/hf-cache', prototype_mode='full-consensus', k_task4=3, k_task6=1, max_train_rows=0, max_test_rows=0, max_desc_tokens=96, eval_mode='both', disable_hierarchical_gate=True)


In [3]:
def build_fewshot_views(
    proto_task4_descs: Dict[str, List[Tuple[str, str]]],
    proto_task6_descs: Dict[str, List[Tuple[str, str]]],
    max_per_label: int = 3,
) -> str:
    lines = []
    lines.append("=== Task4 Few-shot Prototypes ===")
    for label in ("YES", "NO"):
        lines.append(f"[{label}]")
        examples = proto_task4_descs.get(label, [])[:max_per_label]
        for sid, desc in examples:
            lines.append(f"- {sid}: {desc}")
        if not examples:
            lines.append("- (empty)")
        lines.append("")

    lines.append("=== Task6 Few-shot Prototypes ===")
    for label, items in proto_task6_descs.items():
        lines.append(f"[{label}]")
        examples = items[:max_per_label]
        for sid, desc in examples:
            lines.append(f"- {sid}: {desc}")
        if not examples:
            lines.append("- (empty)")
        lines.append("")

    return "\n".join(lines)

In [4]:
splits = create_data_splits(cfg.data_dir, seed=cfg.seed, force_split=cfg.force_split)
train_ids = splits.get("train", [])
test_ids = splits.get("test", [])

train_rows = load_rows(cfg.data_dir, train_ids)
test_rows = load_rows(cfg.data_dir, test_ids)

if cfg.max_train_rows > 0:
    train_rows = train_rows[: cfg.max_train_rows]
if cfg.max_test_rows > 0:
    test_rows = test_rows[: cfg.max_test_rows]

print(f"[data] train/test rows: {len(train_rows)}/{len(test_rows)}")

if cfg.prototype_mode == "full-consensus":
    proto_task4 = select_prototypes_task4_full_consensus(train_rows, k_per_class=cfg.k_task4)
    proto_task6 = select_prototypes_task6_full_consensus(train_rows, k_per_label=cfg.k_task6)
else:
    proto_task4 = select_prototypes_task4(train_rows, k_per_class=cfg.k_task4)
    proto_task6 = select_prototypes_task6(train_rows, k_per_label=cfg.k_task6)

all_proto_ids = set()
for rows in proto_task4.values():
    for r in rows:
        all_proto_ids.add(r.sample_id)
for rows in proto_task6.values():
    for r in rows:
        all_proto_ids.add(r.sample_id)

row_by_id = {r.sample_id: r for r in train_rows + test_rows}
proto_rows = [row_by_id[sid] for sid in all_proto_ids if sid in row_by_id]
desc_rows = proto_rows + test_rows

print(f"[data] prototype rows: {len(proto_rows)}")
print(f"[data] prototype mode: {cfg.prototype_mode}")
print(f"[data] k_task4/k_task6: {cfg.k_task4}/{cfg.k_task6}")

Loading existing data splits from /root/QWen/EXIST 2024 Memes Dataset/data_splits.json
[data] train/test rows: 3235/405
[data] prototype rows: 10
[data] prototype mode: full-consensus
[data] k_task4/k_task6: 3/1


In [5]:
stage1_model_path = resolve_local_model_path(cfg.model_cache_dir, cfg.stage1_model_name)
print(f"[stage1] configured model: {cfg.stage1_model_name}")
print(f"[stage1] resolved model path: {stage1_model_path}")

stage1_processor = AutoProcessor.from_pretrained(
    stage1_model_path,
    trust_remote_code=True,
    cache_dir=cfg.model_cache_dir,
)
stage1_model = AutoModelForImageTextToText.from_pretrained(
    stage1_model_path,
    trust_remote_code=True,
    cache_dir=cfg.model_cache_dir,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
stage1_model.eval()

desc_cache = load_or_generate_descriptions(
    rows=desc_rows,
    desc_cache_path=cfg.desc_cache_path,
    vl_model=stage1_model,
    vl_processor=stage1_processor,
    max_new_tokens=cfg.max_desc_tokens,
)

# 切换到分类器前释放第一阶段模型显存，避免双模型同驻引发 OOM
del stage1_model
del stage1_processor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"[stage1] cached descriptions: {len(desc_cache)}")

[stage1] configured model: Qwen/Qwen3.5-9B
[stage1] resolved model path: /root/autodl-tmp/hf-cache/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a


`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

[stage1] cached descriptions: 415


In [6]:
proto_task4_descs = {
    "YES": [(r.sample_id, desc_cache.get(r.sample_id, "")) for r in proto_task4["YES"]],
    "NO": [(r.sample_id, desc_cache.get(r.sample_id, "")) for r in proto_task4["NO"]],
}
proto_task6_descs = {
    label: [(r.sample_id, desc_cache.get(r.sample_id, "")) for r in rows]
    for label, rows in proto_task6.items()
}

fewshot_preview = build_fewshot_views(
    proto_task4_descs=proto_task4_descs,
    proto_task6_descs=proto_task6_descs,
    max_per_label=5,
 )
print(fewshot_preview)

print("\n=== Agreement Quality (selected prototypes) ===")
for label in ("YES", "NO"):
    print(f"Task4 [{label}]")
    for r in proto_task4[label]:
        print(f"- {r.sample_id}: agreement={task4_agreement(r.labels_task1):.3f}")

for label, rows in proto_task6.items():
    print(f"Task6 [{label}]")
    for r in rows:
        print(f"- {r.sample_id}: agreement={task6_label_agreement(r.labels_task2, label):.3f}")

=== Task4 Few-shot Prototypes ===
[YES]
- 210523: The user wants a brief description of the meme's message, focusing on its sexist implications.

1.  **Analyze the text:** The meme sets up a classic "how many X does it take to change a lightbulb" joke format. The punchline subverts the expectation of a numerical answer by stating, "Don't be silly, feminists can't change anything!"
2.  **Identify the core message:** The joke relies on the stereotype that women
- 210880: The user wants a brief description of the meme's message, focusing on sexism and gender implications.

1.  **Analyze the visual content:**
    *   **Top panel:** A control panel with two buttons. One is labeled "rape" and the other "domestic abuse." A hand is pressing the "domestic abuse" button.
    *   **Bottom panel:** A man is shown in distress, holding his head, looking traumatized or in pain
- 110195: The user wants a brief description of the meme's message, focusing on gender implications.

1.  **Analyze the image

In [7]:
clf_model_path = resolve_local_model_path(cfg.model_cache_dir, cfg.classifier_model_name)
print(f"[classifier] model path: {clf_model_path}")

clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_path, cache_dir=cfg.model_cache_dir)
clf_model = AutoModelForCausalLM.from_pretrained(
    clf_model_path,
    cache_dir=cfg.model_cache_dir,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
clf_model.eval()

metrics = evaluate_pipeline_b(
    test_rows=test_rows,
    desc_cache=desc_cache,
    clf_model=clf_model,
    clf_tokenizer=clf_tokenizer,
    proto_task4_descs=proto_task4_descs,
    proto_task6_descs=proto_task6_descs,
    output_path=cfg.output_dir,
    hierarchical_gate=not cfg.disable_hierarchical_gate,
    eval_mode=cfg.eval_mode,
)

print(f"\n==== Pipeline B ({cfg.stage1_model_name} + Mistral-7B) done ====")
print(f"[config] hierarchical_gate: {not cfg.disable_hierarchical_gate}")
if "hard_hard" in metrics:
    print(f"[hard-hard] Task4 F1(binary): {metrics['hard_hard']['task4']['f1_binary']:.4f}")
    print(f"[hard-hard] Task4 F1(macro):  {metrics['hard_hard']['task4']['f1_macro']:.4f}")
    print(f"[hard-hard] Task6 F1(macro):  {metrics['hard_hard']['task6']['f1_macro']:.4f}")
    print(f"[hard-hard] Task6 F1(micro):  {metrics['hard_hard']['task6']['f1_micro']:.4f}")
    print(f"[hard-hard] Task6 F1(samples):{metrics['hard_hard']['task6']['f1_samples']:.4f}")
if "soft_soft" in metrics:
    print(f"[soft-soft] Task4 Brier:      {metrics['soft_soft']['task4']['brier']:.4f}")
    print(f"[soft-soft] Task6 Brier(macro): {metrics['soft_soft']['task6']['brier_macro']:.4f}")
    print(f"[soft-soft] Task6 Brier(micro): {metrics['soft_soft']['task6']['brier_micro']:.4f}")
print(f"samples: {metrics['num_samples']}")
print(f"metrics path: {os.path.join(cfg.output_dir, 'metrics.json')}")

[classifier] model path: /root/autodl-tmp/mistral_models/7B-Instruct-v0.3


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Classifying: 100%|██████████| 405/405 [06:17<00:00,  1.07it/s]


==== Pipeline B (Qwen/Qwen3.5-9B + Mistral-7B) done ====
[config] hierarchical_gate: False
[hard-hard] Task4 F1(binary): 0.6947
[hard-hard] Task4 F1(macro):  0.6546
[hard-hard] Task6 F1(macro):  0.1816
[hard-hard] Task6 F1(micro):  0.2063
[hard-hard] Task6 F1(samples):0.1058
[soft-soft] Task4 Brier:      0.2410
[soft-soft] Task6 Brier(macro): 0.1483
[soft-soft] Task6 Brier(micro): 0.1483
samples: 405
metrics path: /root/QWen/outputs/pipeline_b_qwen35_mistral/metrics.json
